### RAG Pipelines-Data Ingestion to Vector DB Pipeline

In [2]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\rajes\AppData\Local\Temp\ipykernel_16868\3728471183.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
# Read all the PDF in the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: SQL_1.pdf
  ✓ Loaded 2 pages

Processing: SQL_2.pdf
  ✓ Loaded 6 pages

Total documents loaded: 8


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}, page_content='1. What is SQL?\nSQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate\ndata in relational databases.\n2. What is the difference between INNER JOIN and LEFT JOIN?\n- INNER JOIN returns only matching rows from both tables.\n- LEFT JOIN returns all rows from the left table, and matched rows from the right table (or NULLs if\nno match).\n3. What is the difference between ON and WHERE clause in SQL joins?\n- ON is used to define the join condition between two tables.\n- WHERE is used to filter the final result set after the join is performed.\n4. What is the difference between UNION and UNION ALL?\n- UNION removes duplicate rows.\n- UNION ALL keeps all rows, including dup

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 8 documents into 9 chunks

Example chunk:
Content: 1. What is SQL?
SQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate
data in relational databases.
2. What is the difference between INNER JOIN and LEFT JOIN?...
Metadata: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}, page_content='1. What is SQL?\nSQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate\ndata in relational databases.\n2. What is the difference between INNER JOIN and LEFT JOIN?\n- INNER JOIN returns only matching rows from both tables.\n- LEFT JOIN returns all rows from the left table, and matched rows from the right table (or NULLs if\nno match).\n3. What is the difference between ON and WHERE clause in SQL joins?\n- ON is used to define the join condition between two tables.\n- WHERE is used to filter the final result set after the join is performed.\n4. What is the difference between UNION and UNION ALL?\n- UNION removes duplicate rows.\n- UNION ALL keeps all rows, including dup

### Embedding and VectorStore DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                "description": "PDF document embeddings for RAG",
                "hnsw:space": "cosine"  
            }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 35


In [10]:
### Convert the text to embeddings
### Convert the text to embeddings
texts = [doc.page_content for doc in all_pdf_documents]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
chunk_texts = [doc.page_content for doc in chunks]
chunk_embeddings = embedding_manager.generate_embeddings(chunk_texts)

vectorstore.add_documents(chunks, chunk_embeddings)

Generating embeddings for 8 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (8, 384)
Generating embeddings for 9 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (9, 384)
Adding 9 documents to vector store...
Successfully added 9 documents to vector store
Total documents in collection: 44


### Retriever Pipline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Debug print statement to see what raw numbers ChromaDB is returning
                    print(f"Debug - Doc ID: {doc_id} | Raw Distance: {distance}")
                    
                    # If using L2, similarity metric fallback:
                    similarity_score = 1 - distance  
                    
                    # Switch threshold evaluation or set default score_threshold to a negative float (like -10.0) for testing
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("What is SQL?")

Retrieving documents for query: 'What is SQL?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Debug - Doc ID: doc_f6de9d65_0 | Raw Distance: 0.9780124425888062
Debug - Doc ID: doc_084b41f2_1 | Raw Distance: 0.9959502220153809
Debug - Doc ID: doc_c126ef9e_1 | Raw Distance: 0.9959502220153809
Debug - Doc ID: doc_faf24761_1 | Raw Distance: 0.9959502220153809
Debug - Doc ID: doc_993621ed_1 | Raw Distance: 0.9959502220153809
Retrieved 5 documents (after filtering)


[{'id': 'doc_f6de9d65_0',
  'content': '1. What is SQL?\nSQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate\ndata in relational databases.\n2. What is the difference between INNER JOIN and LEFT JOIN?\n- INNER JOIN returns only matching rows from both tables.\n- LEFT JOIN returns all rows from the left table, and matched rows from the right table (or NULLs if\nno match).\n3. What is the difference between ON and WHERE clause in SQL joins?\n- ON is used to define the join condition between two tables.\n- WHERE is used to filter the final result set after the join is performed.\n4. What is the difference between UNION and UNION ALL?\n- UNION removes duplicate rows.\n- UNION ALL keeps all rows, including duplicates.\n5. What is a Primary Key?\nA Primary Key uniquely identifies each record in a table and does not allow NULLs or duplicates.\n6. What is a Foreign Key?\nA Foreign Key is a column that creates a relationship between two tables by refer

### RAG Pipeline - VectorDB to LLM Output Generation

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [18]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [19]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [20]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Debug - Doc ID: doc_084b41f2_1 | Raw Distance: 1.8658473491668701
Debug - Doc ID: doc_c126ef9e_1 | Raw Distance: 1.8658473491668701
Debug - Doc ID: doc_faf24761_1 | Raw Distance: 1.8658473491668701
Debug - Doc ID: doc_993621ed_1 | Raw Distance: 1.8658473491668701
Debug - Doc ID: doc_0a461225_3 | Raw Distance: 1.87589693069458
Retrieved 0 documents (after filtering)


[]

In [42]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    # 1. Retrieve the context (FIXED: replaced .retrieve with .get_relevant_documents)
    try:
        results = retriever.get_relevant_documents(query)
    except AttributeError:
        # If you later switch to a standard LangChain retriever, use .invoke()
        results = retriever.invoke(query)
        
    # Slice the results to respect top_k
    results = results[:top_k]
    
    # Extract text depending on whether results are dicts or Document objects
    context_list = []
    for doc in results:
        if hasattr(doc, 'page_content'):
            context_list.append(doc.page_content)
        else:
            context_list.append(doc.get('content', doc.get('page_content', '')))
            
    context = "\n\n".join(context_list) if results else ""
    
    if not context:
        return "No relevant context found to answer the question."
    
    # 2. Generate the answer (Using HumanMessage to prevent Groq BadRequestErrors)
    from langchain_core.messages import HumanMessage
    prompt = f"Use the following context to answer the question.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

### Enhanced RAG Pipeline Features

In [37]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # FIX: Use LangChain's standardized method to get documents
    # Note: If your retriever supports configuring top_k dynamically, you can use it here.
    # Otherwise, configure top_k and search_kwargs when initializing your vectorstore retriever.
    try:
        raw_docs = retriever.invoke(query)
    except AttributeError:
        # Fallback for the MockRetriever from previous steps
        raw_docs = retriever.get_relevant_documents(query)

    if not raw_docs:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Process and filter documents based on format (Dict vs LangChain Document object)
    results = []
    for doc in raw_docs:
        # Check if it's a LangChain Document object or a dictionary
        is_obj = hasattr(doc, 'page_content')
        content = doc.page_content if is_obj else doc.get('content', '')
        metadata = doc.metadata if is_obj else doc.get('metadata', {})
        
        # Pull similarity score if available (defaulting to 1.0 if not exposed by retriever)
        score = metadata.get('score', doc.get('similarity_score', 1.0)) if not is_obj else doc.metadata.get('score', 1.0)
        
        if score >= min_score:
            results.append({
                'content': content,
                'metadata': metadata,
                'similarity_score': score
            })

    # Slice results to respect top_k manually if retriever hasn't already
    results = results[:top_k]

    if not results:
        return {'answer': 'No relevant context met the minimum confidence score.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer (FIXED: Wrapped prompt in HumanMessage to avoid previous BadRequestError)
    prompt = f"Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = llm.invoke([HumanMessage(content=prompt)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Techniques", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("\nSources:", result['sources'])
print("\nConfidence:", result['confidence'])
print("\nContext Preview:", result['context'][:300])

Answer: Hard Negative Mining (HNM) techniques are methods used to improve the performance of machine learning models, particularly in object detection and classification tasks. HNM involves selectively sampling and weighting the most informative and difficult-to-classify examples (hard negatives) to update the model, rather than using all available data. This approach helps to reduce the impact of easy-to-classify examples and focuses the model on the most challenging cases, leading to improved accuracy and robustness.

Sources: [{'source': 'unknown', 'page': 'unknown', 'score': 1.0, 'preview': '...'}, {'source': 'unknown', 'page': 'unknown', 'score': 1.0, 'preview': '...'}, {'source': 'unknown', 'page': 'unknown', 'score': 1.0, 'preview': '...'}]

Confidence: 1.0

Context Preview: 






In [34]:
import os
from dotenv import load_dotenv

# 1. Standardized LangChain Imports for 2026 Ecosystem
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables (Make sure your .env file has GROQ_API_KEY)
load_dotenv()

# --- Mock Retriever Class for Demonstration ---
# (Replace this with your actual vector store/Chroma/FAISS retriever)
class MockRetriever:
    def get_relevant_documents(self, query: str):
        return [
            {"page_content": "Attention Is All You Need is a seminal 2017 research paper by Ashish Vaswani et al. that introduced the Transformer architecture."},
            {"page_content": "The Transformer architecture relies entirely on self-attention mechanisms, dispensing with recurrent neural networks (RNNs) or convolutional layers entirely."},
            {"page_content": "Transformers became the foundational architectural standard for modern Large Language Models (LLMs) like GPT, Claude, and Llama."}
        ]

# --- Advanced RAG Pipeline ---
class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, user_query: str, top_k: int = 3, min_score: float = 0.1, stream: bool = False, summarize: bool = False):
        # 1. Retrieve Context
        docs = self.retriever.get_relevant_documents(user_query)
        context = "\n".join([doc["page_content"] for doc in docs[:top_k]])
        
        # 2. Formulate Prompt using Chat System/Human Structure
        system_instruction = "You are a helpful assistant. Answer the user's question accurately using ONLY the provided context."
        user_content = f"Context:\n{context}\n\nQuestion: {user_query}\nAnswer:"
        
        messages = [
            SystemMessage(content=system_instruction),
            HumanMessage(content=user_content)
        ]
        
        # 3. Generate Primary Answer
        # Note: Groq supports ultra-fast streaming, but for standard execution we use invoke here
        response = self.llm.invoke(messages)
        answer = response.content
        
        # 4. Optional Summary Block (FIXED: Uses a properly structured HumanMessage wrapper)
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            # Passing a HumanMessage object inside the list avoids the BadRequestError
            summary_resp = self.llm.invoke([HumanMessage(content=summary_prompt)])
            summary = summary_resp.content
        
        # 5. Track Session History
        self.history.append({"query": user_query, "answer": answer})
        
        return {
            'answer': answer,
            'summary': summary,
            'history': self.history
        }

# --- Execution Setup ---

# Initialize the Groq Chat Model
# Ensure GROQ_API_KEY is present in your environment variables
llm = ChatGroq(
    temperature=0.2, 
    model_name="llama-3.3-70b-versatile",
    api_key=os.environ.get("GROQ_API_KEY")
)

# Instantiate components
rag_retriever = MockRetriever()
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

# Execute the pipeline
result = adv_rag.query(
    user_query="what is attention is all you need", 
    top_k=3, 
    min_score=0.1, 
    stream=True, 
    summarize=True
)

# --- Print Outputs ---
print("\n=== RAG PIPELINE OUTPUT ===")
print("\n[Final Answer]:\n", result['answer'])
print("\n[Summary]:\n", result['summary'])
print("\n[History Tracker]:\n", result['history'][-1])


=== RAG PIPELINE OUTPUT ===

[Final Answer]:
 "Attention Is All You Need" is a 2017 research paper by Ashish Vaswani et al. that introduced the Transformer architecture.

[Summary]:
 The research paper "Attention Is All You Need" was published in 2017 by Ashish Vaswani and colleagues. This paper introduced the Transformer architecture, a significant development in the field of natural language processing and deep learning.

[History Tracker]:
 {'query': 'what is attention is all you need', 'answer': '"Attention Is All You Need" is a 2017 research paper by Ashish Vaswani et al. that introduced the Transformer architecture.'}
